# Multi-Agent Orchestration

Welcome to the final notebook in the Azure AI Agent Service tutorial! This notebook explores advanced patterns for coordinating multiple agents working together.

## Prerequisites

Before starting this tutorial, make sure you've completed:
- **Agent Service Basics** notebook
- **Agent Tools & Capabilities** notebook

You should be familiar with:
- Creating and managing agents
- Using built-in tools (File Search, Code Interpreter)
- Streaming responses

## What You'll Learn

- **Connected Agents**: Using agents as tools for other agents
- **Sequential Workflows**: Pipeline-based processing
- **Agent Handoffs**: Routing conversations to specialists
- **Human-in-the-Loop**: Interactive workflows with user input
- **Orchestration Patterns**: Managing complex multi-agent systems

## Resources
- [Multi-Agent Orchestration Guide](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/agent-orchestration)
- [Connected Agents](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/connected-agents)
- [Foundry Samples - Multi-Agent](https://github.com/azure-ai-foundry/foundry-samples)

Let's build sophisticated multi-agent systems!

## Setup and Imports

In [ ]:
import os
import json
from typing import Optional, Dict, Any
from dotenv import load_dotenv

# Azure AI Projects SDK
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    MessageRole,
    RunStatus,
    FunctionTool,
    CodeInterpreterTool,
    ToolSet,
    MessageDeltaChunk,
    ThreadRun,
    RequiredFunctionToolCall,
    ToolOutput,
    SubmitToolOutputsAction
)
from azure.identity import DefaultAzureCredential

# Load environment variables
load_dotenv()

print("✅ Imports loaded successfully!")

In [ ]:
# Initialize the project client
project_client = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint=os.environ["PROJECT_ENDPOINT"]
)

MODEL_DEPLOYMENT = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o")

print(f"✅ Connected to Azure AI Foundry")
print(f"📦 Using model: {MODEL_DEPLOYMENT}")

## 1. Connected Agents Pattern

One of the most powerful patterns in multi-agent systems is **Connected Agents** - where one agent can call another agent as a tool. This enables:

- **Specialization**: Each agent focuses on what it does best
- **Delegation**: Orchestrator agents route tasks to specialists
- **Composition**: Complex workflows from simple building blocks

### Architecture:
```
User → Orchestrator Agent → Specialist Agent 1
                         → Specialist Agent 2
                         → Specialist Agent 3
```

In [ ]:
# Create specialist agents
# Each agent has a specific expertise

# Research Specialist
research_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="ResearchSpecialist",
    instructions="""You are a research specialist. Your job is to:
    1. Analyze topics and provide comprehensive information
    2. Present facts objectively with multiple perspectives
    3. Structure information clearly with bullet points and sections
    4. Identify key trends and patterns
    
    Always provide well-organized, factual responses."""
)

# Writing Specialist
writing_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="WritingSpecialist",
    instructions="""You are a professional writer. Your job is to:
    1. Transform information into engaging, readable content
    2. Adapt tone and style to the target audience
    3. Create compelling introductions and conclusions
    4. Ensure clarity and flow in all content
    
    Focus on making content accessible and engaging."""
)

# Data Analyst (with code interpreter)
analyst_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="DataAnalyst",
    instructions="""You are a data analyst. Your job is to:
    1. Perform calculations and statistical analysis
    2. Create data visualizations
    3. Identify patterns and insights in data
    4. Present findings clearly with supporting evidence
    
    Always show your work and explain your methodology.""",
    tools=[CodeInterpreterTool()]
)

print("✅ Specialist agents created!")
print(f"   📚 Research: {research_agent.id}")
print(f"   ✍️ Writing: {writing_agent.id}")
print(f"   📊 Analysis: {analyst_agent.id}")

In [ ]:
# Create a helper to run a specialist agent
def call_specialist(agent_id: str, task: str) -> str:
    """Call a specialist agent with a specific task."""
    # Create a temporary thread for this interaction
    thread = project_client.agents.create_thread()
    
    # Add the task as a message
    project_client.agents.create_message(
        thread_id=thread.id,
        role=MessageRole.USER,
        content=task
    )
    
    # Run the agent
    run = project_client.agents.create_and_process_run(
        thread_id=thread.id,
        agent_id=agent_id
    )
    
    # Get response
    messages = project_client.agents.list_messages(thread_id=thread.id)
    response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
    
    # Clean up the temporary thread
    project_client.agents.delete_thread(thread.id)
    
    return response.text.value if response else "No response"

# Wrap specialist calls as functions for the orchestrator
SPECIALIST_AGENTS = {
    "research": research_agent.id,
    "writing": writing_agent.id,
    "analysis": analyst_agent.id
}

def delegate_to_specialist(specialist: str, task: str) -> str:
    """Delegate a task to a specialist agent."""
    if specialist not in SPECIALIST_AGENTS:
        return f"Unknown specialist: {specialist}. Available: {list(SPECIALIST_AGENTS.keys())}"
    
    print(f"  🔄 Delegating to {specialist}...")
    result = call_specialist(SPECIALIST_AGENTS[specialist], task)
    print(f"  ✅ {specialist} completed task")
    return result

# Function map for tool calling
FUNCTIONS = {
    "delegate_to_specialist": delegate_to_specialist
}

print("✅ Specialist delegation functions ready!")

In [ ]:
# Create the Orchestrator Agent
delegation_function = FunctionTool(
    name="delegate_to_specialist",
    description="""Delegate a task to a specialist agent. 
    Specialists available:
    - research: For gathering and analyzing information on topics
    - writing: For creating engaging content from information
    - analysis: For data analysis, calculations, and visualizations""",
    parameters={
        "type": "object",
        "properties": {
            "specialist": {
                "type": "string",
                "enum": ["research", "writing", "analysis"],
                "description": "The specialist to delegate to"
            },
            "task": {
                "type": "string",
                "description": "The specific task for the specialist to complete"
            }
        },
        "required": ["specialist", "task"]
    }
)

orchestrator_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="ProjectOrchestrator",
    instructions="""You are a project orchestrator managing a team of specialists.
    
Your team:
- Research Specialist: Gathers and analyzes information
- Writing Specialist: Creates engaging content
- Data Analyst: Performs calculations and creates visualizations

Your job:
1. Understand the user's request
2. Break it down into subtasks
3. Delegate to appropriate specialists using the delegate_to_specialist function
4. Combine and synthesize the results
5. Present a cohesive final response

Always explain your delegation strategy and synthesize results effectively.""",
    tools=[delegation_function]
)

print(f"✅ Orchestrator agent created!")
print(f"   ID: {orchestrator_agent.id}")

In [ ]:
def run_orchestrator(task: str) -> str:
    """Run a task through the orchestrator with tool handling."""
    # Create thread
    thread = project_client.agents.create_thread()
    
    # Add user message
    project_client.agents.create_message(
        thread_id=thread.id,
        role=MessageRole.USER,
        content=task
    )
    
    # Run with tool handling
    run = project_client.agents.create_run(
        thread_id=thread.id,
        agent_id=orchestrator_agent.id
    )
    
    while run.status in [RunStatus.QUEUED, RunStatus.IN_PROGRESS, RunStatus.REQUIRES_ACTION]:
        run = project_client.agents.get_run(thread_id=thread.id, run_id=run.id)
        
        if run.status == RunStatus.REQUIRES_ACTION:
            if isinstance(run.required_action, SubmitToolOutputsAction):
                tool_outputs = []
                
                for tool_call in run.required_action.submit_tool_outputs.tool_calls:
                    if isinstance(tool_call, RequiredFunctionToolCall):
                        func_name = tool_call.function.name
                        args = json.loads(tool_call.function.arguments)
                        
                        if func_name in FUNCTIONS:
                            result = FUNCTIONS[func_name](**args)
                        else:
                            result = f"Unknown function: {func_name}"
                        
                        tool_outputs.append(
                            ToolOutput(tool_call_id=tool_call.id, output=result)
                        )
                
                run = project_client.agents.submit_tool_outputs_to_run(
                    thread_id=thread.id,
                    run_id=run.id,
                    tool_outputs=tool_outputs
                )
    
    # Get response
    messages = project_client.agents.list_messages(thread_id=thread.id)
    response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
    
    # Cleanup
    project_client.agents.delete_thread(thread.id)
    
    return response.text.value if response else "No response"

print("✅ Orchestrator runner ready!")

In [ ]:
# Test the orchestrator with a complex task
print("🎯 Multi-Agent Task: Content Creation Project")
print("="*60)

task = """Create a brief article about the benefits of renewable energy.
The article should include:
1. Research on current renewable energy trends
2. A calculation showing potential cost savings over 10 years
3. Well-written content suitable for a blog"""

print(f"📝 Task: {task}")
print("\n" + "-"*40 + "\n")

response = run_orchestrator(task)

print("\n" + "="*60)
print("📋 Final Result:")
print(response)

## 2. Sequential Workflow Pattern

In a sequential workflow, the output of one agent becomes the input for the next, creating a pipeline:

```
User Input → Agent 1 → Agent 2 → Agent 3 → Final Output
```

This is useful for:
- Content pipelines (draft → edit → review)
- Data processing (extract → transform → analyze)
- Multi-step reasoning

In [ ]:
# Create a sequential pipeline for content creation

# Step 1: Idea Generator
idea_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="IdeaGenerator",
    instructions="""You generate creative ideas and outlines.
    
    When given a topic:
    1. Generate 3-5 key points to cover
    2. Create a brief outline
    3. Suggest an engaging angle
    
    Output a structured outline that a writer can expand on."""
)

# Step 2: Content Writer
content_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="ContentWriter",
    instructions="""You transform outlines into full content.
    
    When given an outline:
    1. Expand each point into detailed paragraphs
    2. Add examples and explanations
    3. Create smooth transitions between sections
    4. Write an engaging introduction and conclusion
    
    Focus on clarity and engagement."""
)

# Step 3: Editor
editor_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="ContentEditor",
    instructions="""You edit and polish content.
    
    When given content:
    1. Fix any grammar or spelling errors
    2. Improve clarity and flow
    3. Ensure consistent tone
    4. Tighten verbose sections
    5. Add a summary of key changes made
    
    Return the polished content with brief notes on improvements."""
)

print("✅ Pipeline agents created!")
print(f"   💡 Idea Generator: {idea_agent.id}")
print(f"   ✍️ Content Writer: {content_agent.id}")
print(f"   📝 Editor: {editor_agent.id}")

In [ ]:
def run_pipeline(initial_input: str, agents: list, show_intermediate: bool = True) -> str:
    """Run a sequential pipeline through multiple agents."""
    current_input = initial_input
    
    for i, agent in enumerate(agents, 1):
        print(f"\n🔄 Step {i}/{len(agents)}: {agent.name}")
        print("-" * 40)
        
        # Create thread for this step
        thread = project_client.agents.create_thread()
        
        # Add input
        project_client.agents.create_message(
            thread_id=thread.id,
            role=MessageRole.USER,
            content=current_input
        )
        
        # Process
        run = project_client.agents.create_and_process_run(
            thread_id=thread.id,
            agent_id=agent.id
        )
        
        # Get output
        messages = project_client.agents.list_messages(thread_id=thread.id)
        response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
        current_input = response.text.value if response else current_input
        
        if show_intermediate:
            preview = current_input[:300] + "..." if len(current_input) > 300 else current_input
            print(f"📤 Output preview: {preview}")
        
        # Cleanup
        project_client.agents.delete_thread(thread.id)
    
    return current_input

print("✅ Pipeline runner ready!")

In [ ]:
# Test the sequential pipeline
print("🎯 Sequential Pipeline: Content Creation")
print("="*60)

topic = "The impact of artificial intelligence on healthcare"
print(f"📝 Topic: {topic}")

pipeline_agents = [idea_agent, content_agent, editor_agent]
final_content = run_pipeline(topic, pipeline_agents)

print("\n" + "="*60)
print("📋 Final Edited Content:")
print("="*60)
print(final_content)

## 3. Agent Handoff Pattern

The handoff pattern routes conversations to appropriate specialists based on context. This is similar to a customer service system where different specialists handle different types of queries.

```
User → Triage Agent → Technical Support
                   → Billing Support
                   → General Inquiries
```

In [ ]:
# Create support specialists

tech_support = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="TechnicalSupport",
    instructions="""You are a technical support specialist.
    
    Handle issues related to:
    - Software bugs and errors
    - System configurations
    - Technical troubleshooting
    - API and integration questions
    
    Always:
    1. Ask clarifying questions if needed
    2. Provide step-by-step solutions
    3. Explain technical concepts simply
    4. Offer to escalate if you can't resolve"""
)

billing_support = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="BillingSupport",
    instructions="""You are a billing support specialist.
    
    Handle issues related to:
    - Invoices and payments
    - Subscription plans
    - Pricing questions
    - Refunds and credits
    
    Always:
    1. Be empathetic about billing concerns
    2. Explain charges clearly
    3. Offer solutions when possible
    4. Note: You cannot actually process transactions"""
)

general_support = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="GeneralSupport",
    instructions="""You are a general support representative.
    
    Handle:
    - Product information
    - Feature questions
    - Getting started help
    - General inquiries
    
    Be friendly and helpful. Guide users to appropriate resources."""
)

print("✅ Support specialists created!")
print(f"   🔧 Tech Support: {tech_support.id}")
print(f"   💰 Billing Support: {billing_support.id}")
print(f"   ℹ️ General Support: {general_support.id}")

In [ ]:
# Create the routing/triage function
def route_to_specialist(category: str, conversation_context: str) -> str:
    """Route the conversation to the appropriate specialist."""
    specialists = {
        "technical": tech_support.id,
        "billing": billing_support.id,
        "general": general_support.id
    }
    
    if category not in specialists:
        return f"Unknown category: {category}"
    
    print(f"  📞 Transferring to {category} support...")
    
    # Call the specialist
    response = call_specialist(specialists[category], conversation_context)
    
    return response

# Update function map
FUNCTIONS["route_to_specialist"] = route_to_specialist

# Create routing function tool
routing_function = FunctionTool(
    name="route_to_specialist",
    description="""Route the customer to an appropriate specialist based on their issue.
    Categories:
    - technical: Software bugs, errors, API issues, configuration problems
    - billing: Invoices, payments, subscriptions, refunds, pricing
    - general: Product info, features, getting started, other questions""",
    parameters={
        "type": "object",
        "properties": {
            "category": {
                "type": "string",
                "enum": ["technical", "billing", "general"],
                "description": "The type of support needed"
            },
            "conversation_context": {
                "type": "string",
                "description": "Summary of the customer's issue to pass to the specialist"
            }
        },
        "required": ["category", "conversation_context"]
    }
)

# Create triage agent
triage_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="SupportTriage",
    instructions="""You are the first point of contact for customer support.
    
    Your job:
    1. Greet the customer warmly
    2. Understand their issue
    3. Determine the appropriate category (technical, billing, or general)
    4. Route them to the right specialist using route_to_specialist
    
    Categorization guidelines:
    - Technical: bugs, errors, crashes, API problems, configuration
    - Billing: payments, invoices, subscriptions, pricing, refunds
    - General: product questions, features, getting started
    
    Always be friendly and efficient.""",
    tools=[routing_function]
)

print(f"\n✅ Triage agent created!")
print(f"   ID: {triage_agent.id}")

In [ ]:
def handle_support_request(request: str) -> str:
    """Handle a customer support request through triage."""
    thread = project_client.agents.create_thread()
    
    project_client.agents.create_message(
        thread_id=thread.id,
        role=MessageRole.USER,
        content=request
    )
    
    run = project_client.agents.create_run(
        thread_id=thread.id,
        agent_id=triage_agent.id
    )
    
    while run.status in [RunStatus.QUEUED, RunStatus.IN_PROGRESS, RunStatus.REQUIRES_ACTION]:
        run = project_client.agents.get_run(thread_id=thread.id, run_id=run.id)
        
        if run.status == RunStatus.REQUIRES_ACTION:
            if isinstance(run.required_action, SubmitToolOutputsAction):
                tool_outputs = []
                
                for tool_call in run.required_action.submit_tool_outputs.tool_calls:
                    if isinstance(tool_call, RequiredFunctionToolCall):
                        func_name = tool_call.function.name
                        args = json.loads(tool_call.function.arguments)
                        
                        if func_name in FUNCTIONS:
                            result = FUNCTIONS[func_name](**args)
                        else:
                            result = f"Unknown function: {func_name}"
                        
                        tool_outputs.append(
                            ToolOutput(tool_call_id=tool_call.id, output=result)
                        )
                
                run = project_client.agents.submit_tool_outputs_to_run(
                    thread_id=thread.id,
                    run_id=run.id,
                    tool_outputs=tool_outputs
                )
    
    messages = project_client.agents.list_messages(thread_id=thread.id)
    response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
    
    project_client.agents.delete_thread(thread.id)
    
    return response.text.value if response else "No response"

print("✅ Support request handler ready!")

In [ ]:
# Test the handoff system with different types of requests
test_requests = [
    "Hi, I'm getting an error when I try to call the API. It says 'unauthorized'. Can you help?",
    "I was charged twice on my last invoice. Can someone look into this?",
    "What features are included in the Pro plan?"
]

print("🎯 Support Handoff System Test")
print("="*60)

for i, request in enumerate(test_requests, 1):
    print(f"\n📝 Request {i}:")
    print(f"👤 Customer: {request}")
    print("-"*40)
    
    response = handle_support_request(request)
    
    print(f"🤖 Support: {response[:400]}..." if len(response) > 400 else f"🤖 Support: {response}")
    print("\n" + "="*60)

## 4. Human-in-the-Loop Pattern

Sometimes agents need human approval or input before proceeding. This pattern enables:
- Approval workflows
- Human validation of agent decisions
- Collaborative human-AI workflows

In [ ]:
# Simulated human approval function
def get_human_approval(action: str, details: str) -> str:
    """Simulate getting human approval for an action."""
    print(f"\n🔔 APPROVAL REQUIRED")
    print(f"   Action: {action}")
    print(f"   Details: {details}")
    
    # In a real application, this would wait for actual user input
    # For demo purposes, we'll auto-approve with some conditions
    if "delete" in action.lower() or "refund" in action.lower():
        approval = "APPROVED with caution - please document the reason"
    else:
        approval = "APPROVED"
    
    print(f"   ✅ Status: {approval}")
    return approval

FUNCTIONS["get_human_approval"] = get_human_approval

# Create approval request function tool
approval_function = FunctionTool(
    name="get_human_approval",
    description="Request human approval for sensitive actions like refunds, deletions, or major changes",
    parameters={
        "type": "object",
        "properties": {
            "action": {
                "type": "string",
                "description": "The action that requires approval"
            },
            "details": {
                "type": "string",
                "description": "Details about the action and why it's needed"
            }
        },
        "required": ["action", "details"]
    }
)

# Create agent with human-in-the-loop capability
approval_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="ApprovalWorkflowAgent",
    instructions="""You handle customer requests that may require approval.
    
    For the following actions, ALWAYS request human approval first:
    - Refunds over $50
    - Account deletions
    - Plan downgrades
    - Special pricing
    
    Use the get_human_approval function for these requests.
    After getting approval, confirm the action with the customer.
    If not approved, explain alternatives politely.""",
    tools=[approval_function]
)

print(f"✅ Approval workflow agent created!")
print(f"   ID: {approval_agent.id}")

In [ ]:
def run_approval_workflow(request: str) -> str:
    """Run a request through the approval workflow."""
    thread = project_client.agents.create_thread()
    
    project_client.agents.create_message(
        thread_id=thread.id,
        role=MessageRole.USER,
        content=request
    )
    
    run = project_client.agents.create_run(
        thread_id=thread.id,
        agent_id=approval_agent.id
    )
    
    while run.status in [RunStatus.QUEUED, RunStatus.IN_PROGRESS, RunStatus.REQUIRES_ACTION]:
        run = project_client.agents.get_run(thread_id=thread.id, run_id=run.id)
        
        if run.status == RunStatus.REQUIRES_ACTION:
            if isinstance(run.required_action, SubmitToolOutputsAction):
                tool_outputs = []
                
                for tool_call in run.required_action.submit_tool_outputs.tool_calls:
                    if isinstance(tool_call, RequiredFunctionToolCall):
                        func_name = tool_call.function.name
                        args = json.loads(tool_call.function.arguments)
                        
                        if func_name in FUNCTIONS:
                            result = FUNCTIONS[func_name](**args)
                        else:
                            result = f"Unknown function: {func_name}"
                        
                        tool_outputs.append(
                            ToolOutput(tool_call_id=tool_call.id, output=result)
                        )
                
                run = project_client.agents.submit_tool_outputs_to_run(
                    thread_id=thread.id,
                    run_id=run.id,
                    tool_outputs=tool_outputs
                )
    
    messages = project_client.agents.list_messages(thread_id=thread.id)
    response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
    
    project_client.agents.delete_thread(thread.id)
    
    return response.text.value if response else "No response"

print("✅ Approval workflow runner ready!")

In [ ]:
# Test the approval workflow
approval_requests = [
    "I'd like to request a refund of $75 for last month's charge. The service wasn't working.",
    "Can you help me understand my current plan features?",
    "I need to delete my account and all associated data."
]

print("🎯 Human-in-the-Loop Approval Workflow")
print("="*60)

for request in approval_requests:
    print(f"\n👤 Customer: {request}")
    print("-"*40)
    
    response = run_approval_workflow(request)
    
    print(f"\n🤖 Agent: {response}")
    print("\n" + "="*60)

## 5. Parallel Agent Execution

For tasks that can be done concurrently, running agents in parallel improves efficiency.

In [ ]:
import asyncio
import concurrent.futures

def run_agent_task(agent_id: str, task: str, agent_name: str) -> dict:
    """Run a single agent task."""
    result = call_specialist(agent_id, task)
    return {"agent": agent_name, "result": result}

def run_parallel_agents(tasks: list[dict]) -> list[dict]:
    """Run multiple agent tasks in parallel."""
    results = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(tasks)) as executor:
        futures = {
            executor.submit(
                run_agent_task, 
                task["agent_id"], 
                task["task"],
                task["name"]
            ): task["name"]
            for task in tasks
        }
        
        for future in concurrent.futures.as_completed(futures):
            name = futures[future]
            try:
                result = future.result()
                results.append(result)
                print(f"  ✅ {name} completed")
            except Exception as e:
                results.append({"agent": name, "error": str(e)})
                print(f"  ❌ {name} failed: {e}")
    
    return results

print("✅ Parallel execution functions ready!")

In [ ]:
# Test parallel execution
print("🎯 Parallel Agent Execution Test")
print("="*60)

topic = "The future of sustainable transportation"

parallel_tasks = [
    {
        "name": "Research",
        "agent_id": research_agent.id,
        "task": f"Research key trends and developments in: {topic}"
    },
    {
        "name": "Analysis",
        "agent_id": analyst_agent.id,
        "task": f"Analyze potential economic impacts and create a simple projection for: {topic}"
    }
]

print(f"📝 Topic: {topic}")
print(f"🚀 Running {len(parallel_tasks)} agents in parallel...\n")

import time
start_time = time.time()

results = run_parallel_agents(parallel_tasks)

elapsed = time.time() - start_time
print(f"\n⏱️ Completed in {elapsed:.2f} seconds")

print("\n" + "="*60)
print("📋 Results:")
print("="*60)

for result in results:
    print(f"\n🏷️ {result['agent']}:")
    if 'error' in result:
        print(f"   Error: {result['error']}")
    else:
        preview = result['result'][:300] + "..." if len(result['result']) > 300 else result['result']
        print(f"   {preview}")

## 6. Cleanup

In [ ]:
# Clean up all agents created in this notebook
print("🧹 Cleaning up agents...")

all_agents = [
    # Specialist agents
    research_agent.id,
    writing_agent.id,
    analyst_agent.id,
    
    # Orchestrator
    orchestrator_agent.id,
    
    # Pipeline agents
    idea_agent.id,
    content_agent.id,
    editor_agent.id,
    
    # Support agents
    tech_support.id,
    billing_support.id,
    general_support.id,
    triage_agent.id,
    
    # Approval agent
    approval_agent.id
]

for agent_id in all_agents:
    try:
        project_client.agents.delete_agent(agent_id)
        print(f"   ✅ Deleted agent: {agent_id[:20]}...")
    except Exception as e:
        print(f"   ⚠️ Could not delete agent: {e}")

print("\n✅ Cleanup complete!")

## 🎉 Congratulations!

You've completed the Multi-Agent Orchestration tutorial! Here's what you've mastered:

### ✅ Orchestration Patterns:
1. **Connected Agents** - Using agents as tools for orchestrator agents
2. **Sequential Workflows** - Pipeline processing through multiple agents
3. **Agent Handoffs** - Routing to specialists based on context
4. **Human-in-the-Loop** - Approval workflows with human input
5. **Parallel Execution** - Running agents concurrently for efficiency

### 🔧 Key Concepts:
- Orchestrator pattern with delegation
- Pipeline architecture for content workflows
- Triage and routing systems
- Approval and validation workflows
- Concurrent agent execution

### 📊 Pattern Comparison:

| Pattern | Best For | Complexity |
|---------|----------|------------|
| Connected Agents | Complex tasks requiring multiple skills | Medium |
| Sequential | Linear processing pipelines | Low |
| Handoff | Support/routing scenarios | Medium |
| Human-in-the-Loop | Sensitive operations | High |
| Parallel | Independent subtasks | Medium |

### 🚀 What's Next?

You've now completed the Microsoft Agent Framework tutorial series! Here are some ideas for next steps:

1. **Build Real Applications**: Apply these patterns to your own use cases
2. **Explore Advanced Features**: Dive into Azure AI Foundry's enterprise features
3. **Integrate with Other Services**: Connect agents to your existing systems
4. **Production Deployment**: Learn about scaling and monitoring

### 📚 Additional Resources:
- [Azure AI Foundry Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/)
- [Multi-Agent Orchestration Guide](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/agent-orchestration)
- [Azure AI Foundry Samples](https://github.com/azure-ai-foundry/foundry-samples)
- [Microsoft Agent Framework GitHub](https://github.com/microsoft/agent-framework)

Excellent work completing this tutorial series! You're now ready to build sophisticated multi-agent AI systems. 🚀🤖